In [ ]:

import json
from pathlib import Path

import numpy as np
import torch

from datasets import load_dataset
from razdel import sentenize
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)

# НАСТРОЙКИ

MODEL_NAME = "distilbert/distilbert-base-multilingual-cased"
DATASET_NAME = "iitolstykh/LLMTrace_classification"
OUTPUT_DIR = Path("./distilbert_llmtrace_sentences")
MODEL_DIR = OUTPUT_DIR / "model"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_LENGTH = 256
EPOCHS = 2
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
SEED = 42

MAX_TRAIN_DOCUMENTS = 20000
MAX_VAL_SENTENCES = 10_000
MAX_TEST_SENTENCES = 10_000

set_seed(SEED)

# ЗАГРУЗКА ДАТАСЕТА

print("Загружаем исходный датасет...")

raw_dataset = load_dataset(DATASET_NAME)

for split_name, split_data in raw_dataset.items():
    print(f"{split_name}: {len(split_data)} документов")

# ОЧИСТКА ИСХОДНЫХ ДОКУМЕНТОВ

def clean_split(split_data):
    split_data = split_data.select_columns(["text", "label"])

    split_data = split_data.filter(
        lambda row: (
            isinstance(row["text"], str)
            and bool(row["text"].strip())
            and row["label"] in ("human", "ai")
        )
    )

    return split_data


train_docs = clean_split(raw_dataset["train"])
val_docs = clean_split(raw_dataset["validation"])
test_docs = clean_split(raw_dataset["test"])

if (
    MAX_TRAIN_DOCUMENTS is not None
    and MAX_TRAIN_DOCUMENTS < len(train_docs)
):
    train_docs = train_docs.shuffle(seed=SEED).select(
        range(MAX_TRAIN_DOCUMENTS)
    )

print("\nДокументы после очистки:")
print("Train:", len(train_docs))
print("Validation:", len(val_docs))
print("Test:", len(test_docs))

# РАЗБИЕНИЕ НА ПРЕДЛОЖЕНИЯ

def split_into_sentences(batch):
    result = {
        "text": [],
        "label": [],
    }

    for text, label in zip(batch["text"], batch["label"]):
        for sentence in sentenize(text):
            sentence_text = sentence.text.strip()

            if sentence_text:
                result["text"].append(sentence_text)
                result["label"].append(label)

    return result


def convert_to_sentence_dataset(doc_dataset, split_name):
    print(f"\nРазбиваем {split_name} на предложения...")

    sentence_dataset = doc_dataset.map(
        split_into_sentences,
        batched=True,
        batch_size=500,
        remove_columns=doc_dataset.column_names,
        desc=f"Разбиение {split_name} на предложения",
    )

    # Перемешиваем предложения внутри исходного раздела
    sentence_dataset = sentence_dataset.shuffle(seed=SEED)

    print(f"{split_name}: {len(sentence_dataset)} предложений")

    return sentence_dataset


train_ds = convert_to_sentence_dataset(train_docs, "train")
val_ds = convert_to_sentence_dataset(val_docs, "validation")
test_ds = convert_to_sentence_dataset(test_docs, "test")

if (
    MAX_VAL_SENTENCES is not None
    and MAX_VAL_SENTENCES < len(val_ds)
):
    val_ds = val_ds.select(range(MAX_VAL_SENTENCES))

if (
    MAX_TEST_SENTENCES is not None
    and MAX_TEST_SENTENCES < len(test_ds)
):
    test_ds = test_ds.select(range(MAX_TEST_SENTENCES))


print("\nИтоговые размеры:")
print("Train sentences:", len(train_ds))
print("Validation sentences:", len(val_ds))
print("Test sentences:", len(test_ds))

print("\nМетки в train:")
print(train_ds.unique("label"))

print("\nКоличество предложений каждого класса:")
for label in ("human", "ai"):
    count = sum(
        1 for value in train_ds["label"] if value == label
    )
    print(f"{label}: {count}")

# КОДИРОВАНИЕ МЕТОК

label2id = {"human": 0, "ai": 1}
id2label = {0: "HUMAN", 1: "AI"}


def encode_labels(example):
    example["labels"] = label2id[example["label"]]
    return example


train_ds = train_ds.map(encode_labels)
val_ds = val_ds.map(encode_labels)
test_ds = test_ds.map(encode_labels)

train_ds = train_ds.remove_columns(["label"])
val_ds = val_ds.remove_columns(["label"])
test_ds = test_ds.remove_columns(["label"])

# ТОКЕНИЗАЦИЯ

print("\nЗагружаем токенизатор...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )


train_ds = train_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
    desc="Токенизация train",
)

val_ds = val_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
    desc="Токенизация validation",
)

test_ds = test_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
    desc="Токенизация test",
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# ЗАГРУЗКА МОДЕЛИ

print("\nЗагружаем DistilBERT...")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

# МЕТРИКИ

def compute_metrics(eval_pred):
    logits, true_labels = eval_pred

    logits = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    probabilities = exp_logits / exp_logits.sum(
        axis=1, keepdims=True
    )

    ai_probabilities = probabilities[:, 1]
    predictions = np.argmax(logits, axis=1)

    metrics = {
        "accuracy": accuracy_score(true_labels, predictions),
        "precision": precision_score(
            true_labels, predictions,
            pos_label=1, zero_division=0,
        ),
        "recall": recall_score(
            true_labels, predictions,
            pos_label=1, zero_division=0,
        ),
        "f1": f1_score(
            true_labels, predictions,
            pos_label=1, zero_division=0,
        ),
    }

    if len(np.unique(true_labels)) == 2:
        metrics["roc_auc"] = roc_auc_score(
            true_labels, ai_probabilities
        )

    return metrics

# ПАРАМЕТРЫ ОБУЧЕНИЯ

use_gpu = torch.cuda.is_available()

print("\nУстройство:", "GPU" if use_gpu else "CPU")

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    fp16=use_gpu,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    save_total_limit=2,
)

# ОБУЧЕНИЕ

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("\nНачинаем обучение на предложениях...")
trainer.train()

# ОЦЕНКА

print("\nОцениваем лучшую модель на test...")

test_metrics = trainer.evaluate(
    eval_dataset=test_ds,
    metric_key_prefix="test",
)

metrics = {
    key.removeprefix("test_"): float(value)
    for key, value in test_metrics.items()
    if key.startswith("test_")
    and key.removeprefix("test_") in {
        "accuracy", "precision", "recall", "f1", "roc_auc"
    }
}

# СОХРАНЕНИЕ МОДЕЛИ И ОТЧЕТОВ

trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))

with open(
    OUTPUT_DIR / "metrics.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(metrics, file, indent=4, ensure_ascii=False)


report = [
    "LLMTrace — DistilBERT sentence classification",
    "=" * 48,
    f"Модель: {MODEL_NAME}",
    f"Исходных документов train: {len(train_docs)}",
    f"Предложений train: {len(train_ds)}",
    f"Предложений validation: {len(val_ds)}",
    f"Предложений test: {len(test_ds)}",
    f"Максимальная длина: {MAX_LENGTH}",
    f"Количество эпох: {EPOCHS}",
    "",
    "Метрики на тестовых предложениях:",
]

for name in ("accuracy", "precision", "recall", "f1", "roc_auc"):
    if name in metrics:
        report.append(f"{name.upper():10s}: {metrics[name]:.4f}")

with open(
    OUTPUT_DIR / "report.txt",
    "w",
    encoding="utf-8",
) as file:
    file.write("\n".join(report) + "\n")

print("\n" + "\n".join(report))
print(f"\nМодель сохранена: {MODEL_DIR}")
print(f"Отчёт сохранён: {OUTPUT_DIR / 'report.txt'}")
print(f"Метрики JSON: {OUTPUT_DIR / 'metrics.json'}")


Загружаем исходный датасет...
train: 411440 документов
validation: 86696 документов
test: 90950 документов

Документы после очистки:
Train: 20000
Validation: 86696
Test: 90950

Разбиваем train на предложения...


Разбиение train на предложения:   0%|          | 0/20000 [00:00<?, ? examples/s]

train: 240275 предложений

Разбиваем validation на предложения...
validation: 1058850 предложений

Разбиваем test на предложения...


Разбиение test на предложения:   0%|          | 0/90950 [00:00<?, ? examples/s]

test: 1156589 предложений

Итоговые размеры:
Train sentences: 240275
Validation sentences: 10000
Test sentences: 10000

Метки в train:
['ai', 'human']

Количество предложений каждого класса:
human: 116669
ai: 123606


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Оцениваем лучшую модель на test...


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Roc Auc
0.306506,0.638531,2,0.787700,0.755396,0.817895,0.785404,0.875068


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


LLMTrace — DistilBERT sentence classification
Модель: distilbert/distilbert-base-multilingual-cased
Исходных документов train: 20000
Предложений train: 240275
Предложений validation: 10000
Предложений test: 10000
Максимальная длина: 256
Количество эпох: 2

Метрики на тестовых предложениях:
ACCURACY  : 0.7877
PRECISION : 0.7554
RECALL    : 0.8179
F1        : 0.7854
ROC_AUC   : 0.8751

Модель сохранена: distilbert_llmtrace_sentences/model
Отчёт сохранён: distilbert_llmtrace_sentences/report.txt
Метрики JSON: distilbert_llmtrace_sentences/metrics.json


In [ ]:
!pip install razdel